# Zadanie 1, część II: osadzenia słów i tekstów dla anotowanego korpusu PolEmo 2.0

**Word Embeddings (3 pkt)**
1. Osadzenia słów w anotowanym korpusie, co najmniej 2 modele [1 pkt]
2. Wizualizacja t-SNE z odniesieniem do anotacji, wykresy interaktywne [0.5 pkt]
3. Listy k najbardziej podobnych słów i porównanie modeli [1 pkt]
4. Dyskusja wyników [0.5 pkt]

**Text Embeddings (2 pkt)**
1. Osadzenia zdań: FastText, uśrednione Word2Vec, Universal Sentence Encoder [0.5 pkt]
2. Wizualizacja t-SNE dla każdego modelu, wykresy interaktywne [0.5 pkt]
3. Analiza i dyskusja wg pytań pomocniczych [1 pkt]

**Dodatek:** grupowanie (KMeans) słów i zdań oraz klasyfikacja na osadzeniach jako ilościowe uzupełnienie dyskusji ("analiza przydatności do klasyfikacji" z celu ćwiczenia).

**Dane wejściowe:** zbiór z części I, plik `anotacja/zbior_finalny.csv`. Jeśli go jeszcze nie ma, notebook awaryjnie użyje etykiet PolEmo z plików `klucz_*.csv` (bez modyfikatorów) i wyraźnie o tym poinformuje.

**Ważne:** wytrenowane modele zapisują się w folderze `modele/` i są wczytywane przy kolejnych uruchomieniach. Jeśli zmienisz hiperparametry, usuń odpowiedni plik z `modele/`.

In [1]:
import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from datasets import load_dataset
from gensim.models import FastText, KeyedVectors, Word2Vec
from gensim.models.fasttext import FastTextKeyedVectors
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score, f1_score, normalized_mutual_info_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, normalize

SEED = 42
OUT = Path("anotacja")          # wyniki części I
MODELS = Path("modele")         # wytrenowane / pobrane modele
MODELS.mkdir(exist_ok=True)

SENT_LABELS = ["POZ", "NEG", "NEU", "AMB"]
MOD_TYPES = ["WZM", "OSL", "ODW"]
COLORS = {"POZ": "#2ca02c", "NEG": "#d62728", "NEU": "#7f7f7f", "AMB": "#ff7f0e"}
SYMBOLS = {"brak": "circle", "WZM": "triangle-up", "OSL": "triangle-down", "ODW": "x"}

# hiperparametry modeli trenowanych na korpusie
VEC_DIM, WINDOW, MIN_COUNT, EPOCHS = 100, 5, 3, 20
MIN_ANN = 5          # min. liczba zaanotowanych zdań ze słowem (przy mniejszej dominująca etykieta to szum)
ALPHA = 1.0          # wygładzenie (Laplace) przy liczeniu dominującej etykiety słowa
TOPK = 10            # długość list najbardziej podobnych słów

# proste słowa funkcyjne pomijane przy wyborze słów do list sąsiadów
STOP = set("""i w we z ze na do od po za o u a że się nie to jest są był była było były by się
ale lub oraz czy jak tak już też tylko jeszcze przy dla co który która które m ś""".split())

pd.set_option("display.max_colwidth", 140)

## 0. Teoria w pigułce

- **Bag-of-Words / TF-IDF:** zdanie to rzadki wektor o długości słownika. BoW liczy wystąpienia słów, TF-IDF dodatkowo zmniejsza wagę słów występujących w wielu dokumentach. Nie ma tu pojęcia podobieństwa słów: "dobry" i "świetny" to osobne, ortogonalne wymiary.
- **Word2Vec (skip-gram z negative sampling):** każde słowo ma wektor środkowy `v` i wektor kontekstu `u`. Dla pary (słowo, sąsiad z okna) maksymalizujemy `log σ(u·v)`, a dla `k` losowych słów szumu maksymalizujemy `log σ(−u_neg·v)`. Słowa o podobnych kontekstach dostają podobne wektory.
- **FastText:** jak Word2Vec, ale słowo to suma wektorów jego n-gramów znakowych. To ważne dla polskiego, bo fleksja daje wiele form jednego lematu, a rzadkie formy nadal dostają sensowny wektor.
- **GloVe:** faktoryzacja globalnej macierzy współwystępowania. Instrukcja wskazuje go tylko dla angielskiego, więc dla polskiego korpusu go nie trenujemy.
- **Osadzenia zdań:** (a) średnia wektorów słów (Word2Vec, FastText), (b) Universal Sentence Encoder, czyli model uczony bezpośrednio na zdaniach, dający jeden wektor na zdanie.

## Dane

Korpus do trenowania to wszystkie zdania PolEmo 2.0 (ok. 46 tys.), czyli znacznie więcej niż nasza anotowana próbka. Same osadzenia nie wymagają etykiet, więc możemy użyć całości, a anotacje z części I służą do **interpretacji** przestrzeni (kolory, symbole, porównania). Dzięki temu modele mają wystarczająco dużo danych, a ocena odbywa się na naszej anotowanej próbce.

Tokenizacja: małe litery i ciągi znaków `\w+`. Teksty są już stokenizowane, więc formy typu "Była m" zostają rozbite na "była" i "m". Lematyzacji nie robimy, co ma konsekwencje dla polskiej fleksji (omówione w dyskusji).

In [2]:
TOKEN_RE = re.compile(r"\w+", re.UNICODE)


def tokenize(text):
    return TOKEN_RE.findall(str(text).lower())


def load_polemo_texts():
    ds = load_dataset(
        "text",
        data_files="hf://datasets/clarin-pl/polemo2-official/data/all.sentence.train.txt",
    )["train"]
    texts = []
    for line in ds["text"]:
        line = line.strip()
        text = line.rsplit(" __label__", 1)[0] if " __label__" in line else line
        if text.strip():
            texts.append(text.strip())
    return texts


all_texts = load_polemo_texts()
sentences = [tokenize(t) for t in all_texts]
sentences = [s for s in sentences if s]
print(f"Zdań w korpusie do trenowania: {len(sentences)}, tokenów: {sum(map(len, sentences))}")

Zdań w korpusie do trenowania: 45947, tokenów: 690548


Wykazany zbiór 45 947 zdań (ok. 690 tys. tokenów) daje średnio ok. 15 słów na zdanie, co odpowiada typowej długości recenzji internetowej. Taka skala jest w zupełności wystarczająca, aby bez nadzoru wytrenować stabilne modele embeddingowe (Word2Vec, FastText) dla słownictwa wartościującego. Dzięki temu wektory uczą się na dużym korpusie, zanim zostaną zanalizowane na mniejszej, ręcznie zaanotowanej próbce.

### Anotowany zbiór z części I

Wczytujemy `zbior_finalny.csv` (kolumny m.in. `text`, `sentyment`, `modyfikatory`, `gold`). Pomocnicze funkcje do modyfikatorów są skopiowane z części I.

In [3]:
def parse_mods(cell):
    out = []
    for item in str(cell).split(";"):
        item = item.strip()
        if not item:
            continue
        if "=" not in item:
            out.append((item, None, 1))
            continue
        phrase, typ = item.rsplit("=", 1)
        occ = 1
        if "@" in phrase:
            phrase, occ_s = phrase.rsplit("@", 1)
            occ = int(occ_s) if occ_s.strip().isdigit() else 1
        out.append((phrase.strip(), typ.strip().upper(), occ))
    return out


def token_labels(text, cell):
    toks = text.split()
    labels = ["O"] * len(toks)
    for phrase, typ, occ in parse_mods(cell):
        ptoks = [p.lower() for p in phrase.split()]
        if typ not in MOD_TYPES or not ptoks:
            continue
        starts = [i for i in range(len(toks) - len(ptoks) + 1)
                  if [t.lower() for t in toks[i:i + len(ptoks)]] == ptoks]
        if len(starts) < occ:
            continue
        s = starts[occ - 1]
        for j in range(s, s + len(ptoks)):
            labels[j] = typ
    return toks, labels


def load_annotated():
    path = OUT / "zbior_finalny.csv"
    if path.exists():
        d = pd.read_csv(path, index_col=0, dtype=str, keep_default_na=False)
        d = d.rename(columns={"sentyment": "label"})
        print(f"Wczytano {path.name}: {len(d)} zdań")
    else:
        parts = [pd.read_csv(OUT / f"klucz_{n}.csv", dtype=str, keep_default_na=False)
                 for n in ["iter1", "iter2"] if (OUT / f"klucz_{n}.csv").exists()]
        if not parts:
            raise FileNotFoundError("Brak anotacja/zbior_finalny.csv i klucz_*.csv: uruchom najpierw część I.")
        d = pd.concat(parts).set_index("id")
        d["label"] = d["gold"]
        d["modyfikatory"] = ""
        print("UWAGA: brak zbior_finalny.csv, używam etykiet PolEmo z klucz_*.csv (bez modyfikatorów).")
    d = d[d["label"].isin(SENT_LABELS)].copy()
    d["tokens"] = d["text"].apply(tokenize)
    return d


ann = load_annotated()
display(ann[["text", "label", "modyfikatory"]].head())
display(ann["label"].value_counts().reindex(SENT_LABELS).to_frame("liczba zdań"))

Wczytano zbior_finalny.csv: 550 zdań


,text,label,modyfikatory
id,,,
s00000,"Co prawda zabieg jest drogi , cieszę się , że zdecydowała m się na tę biopsję , a nie na operacyjne usunięcie łagodnego guza piersi .",AMB,nie=ODW
s00001,"Co do minusów to myślę , że po prostu ich tu nie znajdziemy .",POZ,nie=ODW
s00002,"ubogie menu , a na pizze trzeba bylo czekac ok 1h .",NEG,
s00003,"Lekarz stara się cały czas dominować nad pacjentem , okazując swą wyższość i traktując go z góry , bardzo nieuprzejmy .",NEG,bardzo=WZM
s00004,"Pani Doktor zajęła się mną holistycznie , opierając się na założeniu , że należy leczyć cały organizm , a nie tylko jego chorą część .",POZ,nie=ODW


,liczba zdań
label,
POZ,177
NEG,213
NEU,120
AMB,40


### Reprezentacja rzadka: BoW i TF-IDF (punkt odniesienia)

Słowa najbardziej wyróżniające każdą klasę naszej anotacji: średnia waga TF-IDF w klasie minus średnia w pozostałych klasach (dzięki temu nie dominują słowa występujące wszędzie). To baza do porównania z osadzeniami: widać, które słowa niosą ocenę, ale model nie wie nic o ich podobieństwie.

In [4]:
bow = CountVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None, min_df=2)
Xb = bow.fit_transform(ann["text"])
tfidf = TfidfVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None, min_df=2)
Xt = tfidf.fit_transform(ann["text"])
print(f"BoW: {Xb.shape}, TF-IDF: {Xt.shape} (zdania x słowa o min_df=2)")

terms = np.array(tfidf.get_feature_names_out())
top_terms = {}
for lab in SENT_LABELS:
    mask = (ann["label"] == lab).to_numpy()
    if mask.sum() == 0 or mask.all():
        continue
    diff = (np.asarray(Xt[mask].mean(axis=0)).ravel()
            - np.asarray(Xt[~mask].mean(axis=0)).ravel())
    top_terms[lab] = terms[np.argsort(-diff)[:10]]
display(pd.DataFrame(top_terms))

BoW: (550, 833), TF-IDF: (550, 833) (zdania x słowa o min_df=2)


,POZ,NEG,NEU,AMB
0,bardzo,nie,które,ale
1,polecam,po,czy,co
2,plus,mnie,się,do
3,dzięki,za,w,dobrej
4,i,ma,także,były
5,smaczne,pokoju,lata,np
6,pokoje,mi,je,nie
7,naprawdę,m,roku,ok
8,położony,nic,kolei,ciepła
9,blisko,przy,system,jeszcze


* **Wymiarowość przestrzeni:** Zastosowanie progu `min_df=2` sprowadziło słownik do 833 unikalnych tokenów dla 550 zdań.


* **Charakterystyka słownictwa w klasach:**
    * **`POZ`:** Dominują bezpośrednie słowa wartościujące i wzmacniacze (*bardzo*, *polecam*, *plus*, *smaczne*, *naprawdę*).


    * **`NEG`:** Wyznaczą ją partykuły przeczące (*nie*, *nic*) oraz zaimki osobiste występujące w opisach negatywnych doświadczeń (*mnie*, *mi*).


    * **`NEU`:** Charakteryzuje się brakiem leksyki emocjonalnej, a dominują w niej spójniki i pojęcia faktograficzne (*które*, *także*, *lata*, *roku*, *system*).


    * **`AMB`:** Na czele listy znajduje się spójnik przeciwstawny *ale*, który wyraźnie sygnalizuje łączenie ocen o odmiennej polaryzacji.

* **Ograniczenie modelu rzadkiego:** TF-IDF poprawnie wyłapuje słowa kluczowe dla klas, jednak traktuje je jako odrębne wymiary (nie uwzględnia faktu, że np. *smaczne* i *dobre* są semantycznie bliskoznaczne).

# Część A. Word Embeddings (3 pkt)

## A1. Osadzenia słów w anotowanym korpusie [1 pkt]

Dwa różne modele trenowane na korpusie, jak wymaga zadanie: **Word2Vec** (skip-gram, negative sampling) i **FastText** (skip-gram z n-gramami znakowymi 3-6). Oba z tymi samymi parametrami okna i wymiaru, żeby porównanie było uczciwe.

Dodatkowo, jeśli pliki są w `modele/`, wczytujemy modele pretrenowane z instrukcji (potrzebne do pierwszego pytania dyskusji):

- Word2Vec dla polskiego: https://dsmodels.nlp.ipipan.waw.pl (plik `word2vec_ipipan.bin` lub inna nazwa, dopasujcie w `PRETRAINED`),
- FastText dla polskiego: https://huggingface.co/clarin-pl/fastText-kgr10 (plik `.bin`, nazwa zawierająca `kgr10`),
- opcjonalnie FastText `cc.pl.300.bin` (https://fasttext.cc/docs/en/crawl-vectors.html).

Jeśli plik nie istnieje, model jest pomijany. Uwaga: niektóre modele IPIPAN mają słowa w postaci lemat::kategoria (np. `dom::subst`); w takim wypadku sprawdźcie pokrycie słownika w tabeli `summary` i w razie potrzeby wybierzcie wariant bez tagów.

In [5]:
def get_model(cls, name, **kwargs):
    path = MODELS / f"{name}.model"
    if path.exists():
        print(f"wczytuję {path.name}")
        return cls.load(str(path))
    print(f"trenuję {name}...")
    m = cls(sentences, vector_size=VEC_DIM, window=WINDOW, min_count=MIN_COUNT,
            sg=1, negative=5, epochs=EPOCHS, seed=SEED, workers=1, **kwargs)
    m.save(str(path))
    return m


w2v = get_model(Word2Vec, "w2v_korpus")
ft = get_model(FastText, "fasttext_korpus", min_n=3, max_n=6)

ALL_MODELS = {
    "W2V (korpus)": w2v.wv,
    "FastText (korpus)": ft.wv,
}

# opcjonalne modele pretrenowane: dostosujcie nazwy plików do pobranych
PRETRAINED = {
    "W2V IPIPAN (pretrenowany)": MODELS / "word2vec_ipipan.bin",
    "FastText kgr10 (pretrenowany)": MODELS / "fasttext_kgr10.bin",
    "FastText cc.pl.300 (pretrenowany)": MODELS / "cc.pl.300.bin",
}


def load_pretrained(path):
    if path.suffix == ".bin" and ("cc." in path.name or "kgr10" in path.name or "fasttext" in path.name.lower()):
        from gensim.models.fasttext import load_facebook_vectors
        return load_facebook_vectors(str(path))
    if path.suffix == ".kv":
        return KeyedVectors.load(str(path))
    return KeyedVectors.load_word2vec_format(str(path), binary=path.suffix == ".bin")


for name, path in PRETRAINED.items():
    if path.exists():
        try:
            ALL_MODELS[name] = load_pretrained(path)
            print(f"wczytano pretrenowany: {name}")
        except Exception as e:
            print(f"nie udało się wczytać {path.name}: {e}")
    else:
        print(f"pomijam (brak pliku {path}): {name}")

summary = pd.DataFrame({
    n: {"słownik": len(kv.key_to_index), "wymiar": kv.vector_size} for n, kv in ALL_MODELS.items()
}).T
display(summary)

trenuję w2v_korpus...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


trenuję fasttext_korpus...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


pomijam (brak pliku modele\word2vec_ipipan.bin): W2V IPIPAN (pretrenowany)
pomijam (brak pliku modele\fasttext_kgr10.bin): FastText kgr10 (pretrenowany)
pomijam (brak pliku modele\cc.pl.300.bin): FastText cc.pl.300 (pretrenowany)


,słownik,wymiar
W2V (korpus),21868,100
FastText (korpus),21868,100


Oba modele (W2V i FastText) wytrenowały słownik obejmujący 21 868 unikalnych tokenów (słów o częstości $\ge 3$) przy jednakowej wymiarowości wektorów równej 100.

Szybki test poprawności: najbliższe sąsiedztwo kilku słów w każdym modelu. Dla FastText działa też słowo spoza słownika (wektor z n-gramów), co jest przewagą nad Word2Vec. Wyniki `topk` są buforowane, bo później liczymy je wielokrotnie (listy, Jaccard, zgodność etykiet).

In [6]:
_TOPK_CACHE = {}


def topk(kv, word, k=TOPK):
    key = (id(kv), word, k)
    if key not in _TOPK_CACHE:
        if word in kv.key_to_index:
            _TOPK_CACHE[key] = [w for w, _ in kv.most_similar(word, topn=k)]
        else:
            _TOPK_CACHE[key] = []
    return _TOPK_CACHE[key]


for test_word in ["hotel", "polecam", "bardzo"]:
    print(f"--- {test_word}")
    for name, kv in ALL_MODELS.items():
        print(f"{name:35s}", ", ".join(topk(kv, test_word, 8)) or "(brak w słowniku)")

oov = "hotelowy"  # forma, która może nie występować w korpusie
print(f"\nFastText, słowo '{oov}' (w słowniku: {oov in ft.wv.key_to_index}):",
      ", ".join(w for w, _ in ft.wv.most_similar(oov, topn=8)))

--- hotel
W2V (korpus)                        pensjonat, obiekt, budynek, hostel, hotelik, ośrodek, położony, apartament
FastText (korpus)                   hotelik, hot, pensjonat, hoteliku, hotelom, hotels, obiekt, hotelowy
--- polecam
W2V (korpus)                        odradzam, polecamy, polecić, polecę, polecała, weekendowe, gorąco, spodoba
FastText (korpus)                   polecamy, polecę, polecać, poleci, polecał, polecana, polecic, polecany
--- bardzo
W2V (korpus)                        dosyć, dość, niezwykle, niezbyt, równie, nadto, naprawdę, dosyc
FastText (korpus)                   baaaardzo, baaardzo, niebardzo, bardo, dość, dosyć, dośc, niezbyt

FastText, słowo 'hotelowy' (w słowniku: True): hotelowi, hostelowy, hotelową, hotelom, hotelowym, hotelowymi, hotelowa, hotelowe


* **Word2Vec** skupia się na podobieństwie kontekstowym i tematycznym – dla słowa *hotel* odnajduje powiązane pojęcia z tej samej kategorii (*pensjonat*, *hostel*, *apartament*). Wykazuje przy tym klasyczną cechę modeli okienkowych: uznaje **antonimy** (*polecam* $\leftrightarrow$ *odradzam*) za wysoce podobne, gdyż pojawiają się w identycznych kontekstach składniowych.
* **FastText** jest silnie zdominowany przez podsłowa (n-gramy znakowe). W efekcie jego najbliższe sąsiedztwo tworzą przede wszystkim formy fleksyjne tego samego wyrazu (*polecamy*, *polecić*, *polecała*, *polecany*).
* Dzięki n-gramom znakowym FastText znakomicie wyłapuje **błędy i ekspresywizmy** charakterystyczne dla recenzji internetowych (np. rozciągnięcia emotywne typu *baaaardzo* czy złączone słowa *niebardzo*).
* FastText skutecznie rozwiązuje problem fleksyjności języka polskiego oraz słów rzadkich/OOV – potrafi zbudować precyzyjny wektor dla odmienionych form (np. *hotelowy*, *hotelowymi*) na podstawie reprezentacji ich fragmentów znakowych.

## A2. Wizualizacja t-SNE słów z odniesieniem do anotacji [0.5 pkt]

### Statystyki słów w anotowanym korpusie

Dla każdego słowa, które wystąpiło w co najmniej `MIN_ANN` zaanotowanych zdaniach, liczymy:

- **dominującą etykietę** z poprawką na rozkład klas. Dla każdej klasy liczymy wygładzony udział zdań ze słowem w tej klasie, `p(klasa | słowo) = (liczba zdań ze słowem w klasie + ALPHA * prior) / (n + ALPHA)`, gdzie `prior` to udział klasy w całym zbiorze, a `n` to liczba zdań ze słowem. Następnie dzielimy go przez prior (**lift**). Lift > 1 oznacza, że słowo występuje w klasie częściej niż wynikałoby z jej liczności, więc liczna klasa (np. NEG) nie wygrywa tylko dlatego, że jest liczna. Wygładzenie ściąga słowa o małym `n` w stronę priora, więc pojedyncze wystąpienie nie daje skrajnego wyniku. `ALPHA` to siła priora (liczba "wirtualnych" zdań). **Czystość** to lift dominującej klasy podzielony przez sumę liftów (poziom przypadkowy to 0.25 przy 4 klasach),
- **typ modyfikatora**, jeśli słowo było oznaczone jako WZM/OSL/ODW w kolumnie `modyfikatory` (dominujący typ spośród oznaczonych wystąpień), w przeciwnym razie `brak`.

Słowa funkcyjne ("i", "w", "jest") nie niosą sentymentu, więc ich czystość jest niska. Dlatego pokazujemy ją też w podpowiedzi na wykresie. Dla bardzo małej klasy (np. AMB) lift jest z natury bardziej zmienny, więc etykiety rzadkich słów są niepewne. Warto to uwzględnić w dyskusji.

In [7]:
class_n = ann["label"].value_counts()
prior = {l: max(class_n.get(l, 0), 1) / len(ann) for l in SENT_LABELS}   # udział klasy w zbiorze

word_labels = defaultdict(Counter)
for toks, lab in zip(ann["tokens"], ann["label"]):
    for w in set(toks):
        word_labels[w][lab] += 1

mod_types = defaultdict(Counter)
for text, cell in zip(ann["text"], ann["modyfikatory"]):
    toks, labs = token_labels(text, cell)
    for t, l in zip(toks, labs):
        if l != "O":
            mod_types[t.lower()][l] += 1

rows = []
for w, cnt in word_labels.items():
    n = sum(cnt.values())
    if n < MIN_ANN:
        continue
    # p(klasa | słowo) wygładzone w stronę priora klasy, potem lift = p / prior
    lift = {l: ((cnt.get(l, 0) + ALPHA * prior[l]) / (n + ALPHA)) / prior[l]
            for l in SENT_LABELS}
    tot = sum(lift.values())
    lab = max(lift, key=lift.get)
    rows.append({
        "word": w, "n": n, "etykieta": lab, "czystość": round(lift[lab] / tot, 2),
        **{f"n_{l}": cnt.get(l, 0) for l in SENT_LABELS},
        "modyfikator": mod_types[w].most_common(1)[0][0] if w in mod_types else "brak",
    })
words_df = pd.DataFrame(rows).sort_values("n", ascending=False).reset_index(drop=True)
print(f"Słów z n >= {MIN_ANN}: {len(words_df)}")
print("Prior klas:", {l: round(p, 3) for l, p in prior.items()})
display(words_df.head(15))
display(words_df["etykieta"].value_counts().reindex(SENT_LABELS).to_frame("słów"))
display(words_df["modyfikator"].value_counts().to_frame("słów"))

Słów z n >= 5: 195
Prior klas: {'POZ': np.float64(0.322), 'NEG': np.float64(0.387), 'NEU': np.float64(0.218), 'AMB': np.float64(0.073)}


,word,n,etykieta,czystość,n_POZ,n_NEG,n_NEU,n_AMB,modyfikator
0,w,177,AMB,0.32,47,70,42,18,OSL
1,i,172,AMB,0.29,63,59,35,15,brak
2,nie,157,AMB,0.37,29,90,20,18,ODW
3,na,152,AMB,0.35,45,62,28,17,WZM
4,z,128,AMB,0.31,37,53,26,12,brak
5,się,121,AMB,0.28,33,49,29,10,brak
6,do,96,AMB,0.44,28,37,16,15,brak
7,to,78,AMB,0.32,22,30,18,8,brak
8,jest,75,AMB,0.30,24,26,18,7,brak
9,że,74,NEG,0.27,21,31,17,5,WZM


,słów
etykieta,
POZ,28
NEG,36
NEU,33
AMB,98


,słów
modyfikator,
brak,159
WZM,21
OSL,12
ODW,3


* **Liczność próby słów:** Kryterium częstotliwości ($n \ge 5$) spełniło **195 unikalnych słów**.


* **Czym jest Lift (współczynnik podbicia):** Mierzy, o ile częściej dane słowo występuje w danej klasie w porównaniu do jej ogólnego udziału (priora) w całym zbiorze. Wartość $\text{lift} > 1$ oznacza nadreprezentację słowa w danej kategorii, co pozwala wyrównać szanse rzadkich klas (np. `AMB`) i zapobiega faworyzowaniu klas najliczniejszych (np. `NEG`).


* **Wpływ priora na słowa funkcyjne:** Najczęstsze tokeny w korpusie to powszechne słowa funkcyjne (*w, i, nie, na, z, się*). Ponieważ klasa `AMB` ma bardzo niski udział bazowy (prior: 7.3%), uniwersalne występowanie tych słów we wszystkich typach zdań powoduje, że wskaźnik lift przypisuje im dominującą etykietę `AMB` (aż 98 słów), czemu towarzyszy jednak niska wartość czystości ($\sim 0.28-0.35$).


* **Separacja słów wartościujących i modyfikatorów:** Wyraziste słowa emocjonalne uzyskują wysoki poziom czystości oraz właściwe etykiety (np. *bardzo* $\rightarrow$ `POZ`, czystość 0.42, modyfikator `WZM`). Większość przeanalizowanych słów nie pełni funkcji modyfikatora (159 tokenów); wśród pozostałych najwięcej jest wzmacniaczy (`WZM`: 21 słów), a znacznie mniej osłabiaczy (`OSL`: 12) i odwracaczy (`ODW`: 3).

### t-SNE słów

Każdy punkt to słowo. **Kolor** = dominująca etykieta zdań, w których się pojawia, **kształt** = typ modyfikatora (kółko oznacza brak). Po najechaniu na punkt widać słowo, liczności etykiet, czystość i typ modyfikatora. Dla wektorów słów używamy odległości kosinusowej (`metric="cosine"`), bo kierunek wektora niesie więcej informacji niż jego długość. Wykresy są zapisywane do `anotacja/tsne_slowa_*.html`.

In [8]:
def run_tsne(X, perplexity=30, metric="euclidean"):
    perplexity = max(2, min(perplexity, (len(X) - 1) / 3))
    return TSNE(n_components=2, perplexity=perplexity, init="pca", metric=metric,
                random_state=SEED, learning_rate="auto").fit_transform(X)


def safe_name(name):
    return re.sub(r"\W+", "_", name).strip("_").lower()


def plot_words(name, kv):
    sub = words_df[words_df["word"].isin(kv.key_to_index)].copy()
    if len(sub) < 5:
        print(f"{name}: za mało słów w słowniku ({len(sub)}), pomijam t-SNE")
        return
    X = np.vstack([kv[w] for w in sub["word"]])
    XY = run_tsne(X, metric="cosine")
    sub["x"], sub["y"] = XY[:, 0], XY[:, 1]
    fig = px.scatter(
        sub, x="x", y="y", color="etykieta", symbol="modyfikator",
        color_discrete_map=COLORS, symbol_map=SYMBOLS,
        category_orders={"etykieta": SENT_LABELS, "modyfikator": list(SYMBOLS)},
        hover_name="word",
        hover_data={"x": False, "y": False, "n": True, "czystość": True,
                    "n_POZ": True, "n_NEG": True, "n_NEU": True, "n_AMB": True},
        title=f"t-SNE słów: {name} ({len(sub)} słów, kolor = etykieta, kształt = modyfikator)",
        width=950, height=700,
    )
    fig.update_traces(marker=dict(size=9, line=dict(width=0.5, color="white")))
    fig.write_html(OUT / f"tsne_slowa_{safe_name(name)}.html")
    fig.show()


for name, kv in ALL_MODELS.items():
    plot_words(name, kv)

* **Brak separacji klas sentymentu:** Zarówno na wykresie Word2Vec, jak i FastText, punkty reprezentujące słowa o różnej dominującej etykiecie (`POZ` – zielone, `NEG` – czerwone, `NEU` – szare) są ze sobą silnie przemieszane. Nie obserwujemy wyraźnych, odseparowanych klastrów odpowiadających polaryzacji emocjonalnej.


* **Przyczyna (Natura modeli bez nadzoru):** Klasyczne osadzenia słów uczą się **podobieństwa kontekstowego i składniowego**, a nie ładunku emocjonalnego. Słowa o przeciwnej polaryzacji (np. *dobry* i *zły* lub *polecam* i *odradzam*) występują w niemal identycznych oknach kontekstowych, przez co lądują bardzo blisko siebie w przestrzeni 2D.


* **Dominacja słów funkcyjnych i modyfikatorów:** Pomarańczowe punkty (`AMB`, obejmujące większość słów funkcyjnych z uwagi na wygładzenie lift) tworzą gęste tło w całym centrum wykresu. Modyfikatory (trójkąty i iksy) nie tworzą własnej osobnej podprzestrzeni, lecz leżą w sąsiedztwie słów, z którymi współwystępują w zdaniach.


* **Wpływ kontekstu próby na etykiety słów:** Przykładowe podświetlone słowo *„dziecko”* uzyskało etykietę `NEG` (czystość 0.65, 5 z 6 wystąpień w zdaniach negatywnych). Pokazuje to, że etykieta słowa w tej analizie odzwierciedla specyfikę tematyczną badanej próby recenzji (np. skargi na opieki medyczną lub pobyt w hotelu), a nie inherentną emocjonalność samych pojęć.

## A3. Listy k najbardziej podobnych słów i porównanie modeli [1 pkt]

Wybieramy słowa **zaanotowane w naszym korpusie**, w trzech grupach:

- najczęstsze **modyfikatory** (WZM/OSL/ODW) z kolumny `modyfikatory`,
- najczęstsze słowa o **wyraźnie pozytywnym** i **wyraźnie negatywnym** rozkładzie etykiet (`n` >= 5, bez słów funkcyjnych z listy `STOP` i bez słów krótszych niż 3 znaki). Wymagamy czystości >= 0.7, ale jeśli przy tym progu jest mniej niż 5 kandydatów w grupie, próg jest automatycznie obniżany (0.6, 0.5, 0.4, a w ostateczności bez progu). Użyty próg jest wypisywany. Przy małej próbce anotacji (kilkaset zdań) obniżenie progu jest normalne, ale warto o tym wspomnieć w dyskusji.

Dla każdego słowa wypisujemy `TOPK` najbliższych sąsiadów (podobieństwo cosinusowe) w każdym modelu.

In [9]:
THRESHOLDS = [0.7, 0.6, 0.5, 0.4, 0.0]
MIN_GROUP = 5          # tyle słów w grupie uznajemy za wystarczające

selected = []   # (słowo, grupa)
mods = words_df[words_df["modyfikator"] != "brak"].head(8)
selected += [(w, f"modyfikator {m}") for w, m in zip(mods["word"], mods["modyfikator"])]

base = words_df[(words_df["n"] >= 5) & (~words_df["word"].isin(mods["word"]))
                & (~words_df["word"].isin(STOP)) & (words_df["word"].str.len() >= 3)]
for lab, grp in [("POZ", "słowo POZ"), ("NEG", "słowo NEG")]:
    cand = base[base["etykieta"] == lab]
    for thr in THRESHOLDS:
        sub = cand[cand["czystość"] >= thr]
        if len(sub) >= MIN_GROUP:
            break
    sub = sub.head(8)    # words_df jest posortowany malejąco po n
    print(f"{grp}: użyty próg czystości {thr}, wybrano {len(sub)} słów")
    selected += [(w, grp) for w in sub["word"]]
print(f"Wybrano {len(selected)} słów do porównania")

rows = []
for w, grp in selected:
    row = {"słowo": w, "grupa": grp}
    for name, kv in ALL_MODELS.items():
        nb = topk(kv, w)
        row[name] = ", ".join(nb) if nb else "(brak w słowniku)"
    rows.append(row)
knn_table = pd.DataFrame(rows)
display(knn_table)
knn_table.to_csv(OUT / "najblizsi_sasiedzi.csv", index=False, encoding="utf-8-sig")

słowo POZ: użyty próg czystości 0.6, wybrano 6 słów
słowo NEG: użyty próg czystości 0.6, wybrano 7 słów
Wybrano 21 słów do porównania


,słowo,grupa,W2V (korpus),FastText (korpus)
0,w,modyfikator OSL,"kaloryfery, muzycznego, skomentowała, grzały, gospody, sportowego, padają, iście, najbliższym, preparatami","sasiednim, bielańskim, sosnowcu, świętokrzyskim, izraelu, irlandii, cranovsky, olimpijskich, wschodnim, szwajcarii"
1,nie,modyfikator ODW,"śmiać, niepokojącego, sens, wiatraka, uważasz, odpisać, głodnym, młodsza, gadac, łzs","masakrycznie, wgl, neutralnie, kompletnie, obojętnie, totalnie, niewyraźnie, nikogo, zbytnio, łzs"
2,na,modyfikator WZM,"końców, znali, groźnie, zaparcia, ucieszył, dostaniesz, stoją, i, umowie, sekundy","żółwie, i, ucieknie, hsg, sceptycznie, potwornie, krajobraz, statystycznie, niezwłocznie, pochopnie"
3,że,modyfikator WZM,"iż, rację, zoperować, osłonę, planuję, zrobią, powtórzy, niestosowne, cytuję, utrzyma","iż, niemalże, niewyraźnie, próśb, śliż, wcale, zgroza, iz, niedosłuch, cykle"
4,bardzo,modyfikator WZM,"dosyć, dość, niezwykle, niezbyt, równie, nadto, naprawdę, dosyc, baaardzo, całkiem","baaaardzo, baaardzo, niebardzo, bardo, dość, dosyć, dośc, niezbyt, dosyc, wyjątkowo"
5,jak,modyfikator WZM,"jakby, dlaczego, identycznie, głupoty, intruz, uciec, przypominam, jakich, git, biedny","jakby, jakos, jakimiś, jakub, masowo, jakoś, nijak, kompleksowo, jakichś, przedmiotowo"
6,po,modyfikator OSL,"prostu, tygodniach, dniach, zaczynamy, miesiącach, trwającym, minutach, używania, następnego, wyraźną","poprostu, prostu, porostu, męczarni, miesiącach, gwóźdź, powrocie, rosyjsku, opuchlizna, miesiącu"
7,o,modyfikator WZM,"chodzi, ryzyku, świadczy, informacji, posprzątanie, zagrożeniu, winien, przepuklinę, skutkach, prawdy","informują, chodzi, informując, formule, wielokrotnie, franczyzy, formacie, swiadczy, informować, świadczy"
8,polecam,słowo POZ,"odradzam, polecamy, polecić, polecę, polecała, weekendowe, gorąco, spodoba, serdecznie, współczuję","polecamy, polecę, polecać, poleci, polecał, polecana, polecic, polecany, polecane, polecała"
9,plus,słowo POZ,"oceniamy, minus, przemawia, jedyny, plusy, korek, łózko, ukłon, późne, pochwalę","plusa, plusy, pluskwy, minus, plusem, plusów, minusik, minusy, jedyny, minusa"


1. Metodologia doboru słów i adaptacja progu czystości
- Wybór 21 słów reprezentatywnych opierał się na trzech grupach zaanotowanych w korpusie PolEmo 2.0: najczęstszych modyfikatorach (WZM, OSL, ODW) oraz wyrazach o wysokim zagęszczeniu wydźwięku pozytywnego (POZ) lub negatywnego (NEG).
- Adaptacja progu czystości: Ze względu na ograniczony rozmiar próby zaanotowanej (550 zdań), sztywny próg czystości $\text{lift} \ge 0.7$ wykluczyłby zbyt wiele kluczowych pojęć. Algorytm automatycznie obniżył próg do $0.6$, co pozwoliło wyselekcjonować 6 reprezentatywnych słów dla grupy POZ oraz 7 dla grupy NEG.
- Filtracja szumu: Wykluczenie słów funkcyjnych z listy STOP oraz tokenów krótszych niż 3 znaki zagwarantowało, że porównanie skupia się na leksyce niosącej realne znaczenie semantyczne lub emocjonalne.

2. Analiza grupowa i przypadki szczególne
- **A. Modyfikatory (WZM, OSL, ODW)** - **`bardzo` / `nie` / `po`:** *Word2Vec* grupuje słowa o podobnej roli i kolokacje (*dosyć*, *dniach*), podczas gdy *FastText* wyłapuje błędy, rozciągnięcia i formy potoczne (*baaaardzo*, *poprostu*).

- **B. Słowa pozytywne (POZ)** - **`polecam` / `położony` / `smaczne`:** *Word2Vec* tworzy spójny kontekst i synonimy (*usytuowany*, *pyszne*), ale myli się przy antonimach (*odradzam*). *FastText* generuje głównie odmianę (*polecamy*) oraz zaprzeczenia z n-gramów (*niesmaczne*).

- **C. Słowa negatywne (NEG)** - **`cena` / `wizytę` / `dziecko`:** *Word2Vec* odtwarza relacje dziedzinowe i wartościujące (*wygórowana*, *konsultację*), a *FastText* stawia na odmianę (*wizytą*) i zbieżności rdzeniowe (*scena*).

3. Wykryte ograniczenia
Wykorzystanie sub-wordów w FastText powoduje powstawanie błędnych relacji semantycznych dla słów, które dzielą przypadkowe ciągi znaków:
- plus $\rightarrow$ pluskwy (przypadkowy prefiks plus- połączony z tematyką hotelową).
- razem $\rightarrow$ obrazem, zarazem, mężem (dopasowanie końcówki -zem).
- wody $\rightarrow$ dowody, przewody (dopasowanie ciągu -wody).
- siebie $\rightarrow$ niebieski, niebieskie (dopasowanie n-gramów -iebi-).


4. Wnioski w kontekście przydatności do klasyfikacji sentymentu
- Word2Vec lepiej chwyta abstrakcyjne relacje semantyczne i kontekst domenowy, jednak wymaga wsparcia w postaci lematyzacji (aby uniknąć rozpraszania masy prawdopodobieństwa na dziesiątki form odmienionych) oraz słownika negacji / antonimów.

- FastText nie wymaga wstępnej lematyzacji i świetnie radzi sobie z bogatą polską fleksją oraz błędami użytkowników, lecz ma tendencję do mieszania polaryzacji poprzez doklejanie słów z przedrostkami negującymi (smaczne / niesmaczne) lub o podobnej budowie morfologicznej.

### Ilościowe porównanie modeli

- **Zgodność list (Jaccard@k):** jaka część sąsiadów jest wspólna dla pary modeli (średnia po słowach obecnych w obu).
- **Zgodność z anotacją:** wśród sąsiadów, które same występują w anotacjach (`n` >= `MIN_ANN`), jaki odsetek ma tę samą dominującą etykietę co słowo-zapytanie. Poziom losowy to prawdopodobieństwo, że **losowe** zaanotowane słowo ma tę samą etykietę co słowo-zapytanie (uśrednione po faktycznie użytych zapytaniach).

In [10]:
def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if (a or b) else np.nan


names = list(ALL_MODELS)
jac = pd.DataFrame(index=names, columns=names, dtype=float)
for m1 in names:
    for m2 in names:
        vals = []
        for w, _ in selected:
            a, b = topk(ALL_MODELS[m1], w), topk(ALL_MODELS[m2], w)
            if a and b:
                vals.append(jaccard(a, b))
        jac.loc[m1, m2] = np.mean(vals) if vals else np.nan
print(f"Średni Jaccard@{TOPK} między listami sąsiadów:")
display(jac.round(3))

dom = words_df.set_index("word")["etykieta"]
share = words_df["etykieta"].value_counts(normalize=True)
query_words = [w for w, g in selected if g.startswith("słowo")]
# poziom losowy: prawdopodobieństwo, że LOSOWE zaanotowane słowo ma tę samą etykietę
# co słowo-zapytanie (uśrednione po faktycznie użytych zapytaniach)
chance = float(np.mean([share.get(dom[w], 0.0) for w in query_words])) if query_words else np.nan


def label_agreement(kv, words):
    hit = tot = 0
    for w in words:
        for nb in topk(kv, w):
            if nb in dom.index:
                tot += 1
                hit += int(dom[nb] == dom[w])
    return (hit / tot if tot else np.nan), tot


agree = {}
for name, kv in ALL_MODELS.items():
    acc, tot = label_agreement(kv, query_words)
    agree[name] = {"odsetek zgodnych etykiet": round(acc, 3), "liczba par": tot}
agree_df = pd.DataFrame(agree).T
agree_df["poziom losowy"] = round(chance, 3)
display(agree_df)

Średni Jaccard@10 między listami sąsiadów:


,W2V (korpus),FastText (korpus)
W2V (korpus),1.000,0.127
FastText (korpus),0.127,1.000


,odsetek zgodnych etykiet,liczba par,poziom losowy
W2V (korpus),0.0,1.0,0.166
FastText (korpus),0.5,4.0,0.166


* **Niska zgodność (Jaccard = 0,127):** Odmienna natura modeli sprawia, że ich listy sąsiadów pokrywają się tylko w ~13% (Word2Vec bazuje na kontekście, FastText na n-gramach).


* **Ograniczona próba:** Ze względu na znikomą liczbę nakładających się par (1 dla W2V, 4 dla FastText) wyniki procentowe mają charakter jedynie poglądowy.


* **Zgodność etykiet:** FastText (0,500 vs 0,166 losowo) przewyższa przypadek, łącząc formy fleksyjne o tej samej polaryzacji, podczas gdy Word2Vec (0,000) zbliża do siebie pojęcia kontekstowe i antonimy.

### Grupowanie słów (KMeans) a anotacja

Dotąd skupiska sentymentu oceniamy wzrokowo (t-SNE) i pośrednio (zgodność etykiet sąsiadów). Tu liczymy to wprost: grupujemy wektory słów algorytmem KMeans (k = 4, tyle ile klas) i porównujemy podział z dominującą etykietą słowa za pomocą **ARI** i **NMI** (obie miary są skorygowane o przypadek, wartość bliska 0 oznacza brak związku z etykietami). Robimy to dla dwóch zbiorów słów:

- **wszystkie** słowa z `n` >= `MIN_ANN`,
- **wyraźne**: 25% słów o najwyższej czystości, bez słów z listy `STOP` i krótszych niż 3 znaki.

Jeśli ARI/NMI są bliskie zera nawet dla wyraźnych słów, to potwierdza, że osadzenia grupują słowa raczej tematycznie niż wg sentymentu.

In [11]:
def word_cluster_scores(kv, df, k=4):
    sub = df[df["word"].isin(kv.key_to_index)]
    if len(sub) < k * 5 or sub["etykieta"].nunique() < 2:
        return None
    X = normalize(np.vstack([kv[w] for w in sub["word"]]))
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X)
    return (len(sub), adjusted_rand_score(sub["etykieta"], km.labels_),
            normalized_mutual_info_score(sub["etykieta"], km.labels_))


clear = words_df[(~words_df["word"].isin(STOP)) & (words_df["word"].str.len() >= 3)]
clear = clear[clear["czystość"] >= clear["czystość"].quantile(0.75)]
subsets = {"wszystkie słowa": words_df, "wyraźne słowa": clear}

rows = {}
for name, kv in ALL_MODELS.items():
    row = {}
    for sname, df_ in subsets.items():
        r = word_cluster_scores(kv, df_)
        if r is None:
            row.update({f"{sname}: liczba": np.nan, f"{sname}: ARI": np.nan, f"{sname}: NMI": np.nan})
        else:
            row.update({f"{sname}: liczba": r[0], f"{sname}: ARI": round(r[1], 3), f"{sname}: NMI": round(r[2], 3)})
    rows[name] = row
display(pd.DataFrame(rows).T)

C:\Users\pauko\anaconda3\envs\CONDA_ENVIRONMENT\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\pauko\anaconda3\envs\CONDA_ENVIRONMENT\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\pauko\anaconda3\envs\CONDA_ENVIRONMENT\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\pauko\anaconda3\envs\CONDA_ENVIRONMENT\Lib\site-packages\s

,wszystkie słowa: liczba,wszystkie słowa: ARI,wszystkie słowa: NMI,wyraźne słowa: liczba,wyraźne słowa: ARI,wyraźne słowa: NMI
W2V (korpus),195.0,0.087,0.058,35.0,0.049,0.170
FastText (korpus),195.0,0.080,0.069,35.0,0.006,0.154


* **Wartości miar ARI i NMI bliskie zera:** Wyniki ARI na poziomie $0{,}080$–$0{,}087$ dla wszystkich słów oraz $0{,}006$–$0{,}049$ dla słów wyraźnych wskazują na brak istotnego powiązania klastrów wyznaczonych przez KMeans z etykietami sentymentu.


* **Brak poprawy po ograniczeniu do wyrazistych słów:** Zwężenie zbioru do 35 słów o najwyższej czystości (po odrzuceniu słów funkcyjnych) nie podnosi wskaźnika ARI, a jedynie nieznacznie zwiększa miarę NMI (do $0{,}170$ dla W2V i $0{,}154$ dla FastText).


* **Potwierdzenie natury osadzeń beznadzorczych:** Wyniki te stanowią bezpośredni dowód na to, że beznadzorcze osadzenia słów (Word2Vec i FastText) grupują pojęcia na podstawie współwystępowania tematycznego i relacji składniowych, a nie pod kątem ładunku emocjonalnego/polaryzacji.

## A4. Dyskusja wyników: Word Embeddings [0.5 pkt]

**1. Czy osadzenia pretrenowane na dużych korpusach dobrze odwzorowują znaczenie słów w naszym korpusie dziedzinowym?**
Wskazówki: pokrycie słownika (tabela `summary`), jakość sąsiadów dla słów z domeny recenzji (hotel, lekarz, polecam), Jaccard między modelami pretrenowanymi a trenowanymi na PolEmo. Jeśli nie dodaliście modeli pretrenowanych, napiszcie to wprost i oceńcie, czego brakuje do odpowiedzi.

- Podczas uruchomienia skryptu pliki modeli zewnętrznych (word2vec_ipipan.bin, fasttext_kgr10.bin, cc.pl.300.bin) nie znajdowały się w katalogu modele/ i zostały pominięte. Analiza opiera się wyłącznie na modelach wytrenowanych bezpośrednio na korpusie PolEmo 2.0 (słownik: 21 868 tokenów, wymiarowość: 100).

- Do oceny modeli pretrenowanych brakuje porównania pokrycia słownika w tabeli summary, wyznaczenia wskaźników Jaccard@k między modelami ogólnymi a korpusowymi oraz analizy jakości sąsiadów dla kluczowych słów dziedzinowych.

- Modele wytrenowane na korpusie PolEmo dobrze uchwyciły specyficzny kontekst recenzji (np. dla słowa wizytę model W2V odnajduje kontrolną, prywatną, konsultację, 120zł, a dla słowa dziecko — symptomatykę medyczną gorączkowało, miesięczne). Modele pretrenowane na ogólnych zasobach (np. Wikipedia, Common Crawl) oferują większy słownik, ale mogą gubić wąskie zależności dziedzinowe i specyficzny słownik wartościujący.

**2. Czy w korpusie występuje grupowanie słów związanych z anotacją? Czy słowa pozytywne i negatywne tworzą osobne skupiska? Jeśli nie, dlaczego osadzenia mogą nie oddawać emocjonalnego znaczenia słów?**
Wskazówki: skupiska sentymentu czy tematu na wykresach t-SNE, tabela zgodności etykiet sąsiadów vs poziom losowy, wyniki KMeans (ARI/NMI) dla słów, antonimy ("dobry" i "zły" występują w tych samych kontekstach, więc mają podobne wektory), położenie modyfikatorów WZM/OSL/ODW (własne skupisko jako słowa funkcyjne czy razem ze słowami, które modyfikują). Pamiętajcie, że etykieta słowa jest szacowana z kilkuset zdań, więc dla rzadkich słów jest niepewna.

* **Brak separacji polaryzacji (t-SNE i KMeans):** Słowa o etykietach `POZ`, `NEG` i `NEU` są silnie zmieszane na wykresach, co potwierdzają wartości ARI ($0{,}006$–$0{,}087$) i NMI ($0{,}058$–$0{,}170$) bliskie zera dla grupowania KMeans ($k=4$). Osadzenia tworzą klastry tematyczne i kontekstowe, a nie emocjonalne.
* **Hipoteza dystrybucyjna i problem antonimów:** Algorytm Skip-gram bazuje na otoczeniu słów, przez co pojęcia o identycznych schematach składniowych, lecz przeciwnym ładunku (*polecam* $\leftrightarrow$ *odradzam*, *dobry* $\leftrightarrow$ *zły*), lądują bardzo blisko siebie w przestrzeni wektorowej.
* **Rozproszenie modyfikatorów i kontekstowość etykiet:** Modyfikatory (`WZM`, `OSL`, `ODW`) nie tworzą własnego skupiska, lecz leżą przy swoich kolokacjach, a etykiety wynikają ze specyfiki badanych recenzji (np. *dziecko* jako `NEG` przez skargi na opieki zdrowotnej/hotele), a nie ze stałej cechy wyrazów.

**3. Na czym polegają różnice między listami podobieństwa różnych modeli i z czego wynikają?**
Wskazówki: FastText zwraca częściej inne formy fleksyjne tego samego słowa (n-gramy znakowe), Word2Vec bardziej zbliża słowa o podobnym kontekście (sprawdźcie, czy to widać w waszych tabelach); wpływ braku lematyzacji; rzadkie słowa mniej stabilne; macierz Jaccarda.

- Niska zgodność sąsiadów (Jaccard@10 = 0,127): Listy 10 najbliższych sąsiadów w Word2Vec i FastText pokrywają się jedynie w ~13%, co wynika z odmiennej architektury.
- Word2Vec (Kontekst semantyczny): Skupia się na relacjach pojęciowych i synonimach (położony $\rightarrow$ usytuowany, zlokalizowany; smaczne $\rightarrow$ świeżutkie, pyszne). Ze względu na brak lematyzacji gubi jednak powiązania między różnymi formami tego samego słowa i silnie przyciąga antonimy (polecam $\rightarrow$ odradzam).
- FastText (Sub-wordy / n-gramy znakowe 3–6): Zdominowany przez budowę morfologiczną. Doskonale radzi sobie z bogatą polską fleksją (polecam $\rightarrow$ polecamy, polecę, polecać), błędami pisowni i ekspresywizmami (bardzo $\rightarrow$ baaaardzo, bardo). Wpada jednak w pułapki dopasowania powierzchniowego, łącząc słowa o zupełnie innym znaczeniu (plus $\rightarrow$ pluskwy, razem $\rightarrow$ obrazem) lub z przedrostkiem negującym (smaczne $\rightarrow$ niesmaczne).

**4. Jak moglibyśmy ulepszyć osadzenia dla rozwiązywanego zadania?**
Wskazówki: lematyzacja, więcej epok i większy korpus, dostrajanie modeli pretrenowanych na PolEmo, osadzenia kontekstowe (np. HerBERT), osadzenia uwzględniające sentyment (retrofitting), obsługa negacji.

- Lematyzacja w preprocessingu: Sprowadzenie słów do form podstawowych (np. Stanza/Spacy) zapobiegnie rozpraszaniu statystyk na dziesiątki form odmienionych w Word2Vec oraz wyelimuje powtarzalność odmian na listach sąsiadów w FastText.

- Retrofitting i osadzenia nadzorowane (Senti2Vec): Wykorzystanie zewnętrznych leksykonów wydźwięku do post-processingu wektorów (modyfikacja przestrzeni tak, aby odpychać od siebie wektory antonimów i przyciągać słowa o tej samej polaryzacji).

- Powiększenie korpusu i fine-tuning: Wytrenowanie modeli na większym zbiorze recenzji branżowych (np. Ceneo, Tripadvisor) lub dostrojenie gotowych dużych modeli (kgr10 / IPIPAN) do specyfiki korpusu PolEmo.

- Osadzenia kontekstowe (Transformery, np. HerBERT / PolBERT): Zastąpienie statycznych wektorów modelami głębokimi, co pozwala dynamicznie reprezentować znaczenie słowa w zależności od całego zdania i natywnie obsługiwać negacje (np. rozróżniać kontekst "polecam" od "nie polecam").

**Uwaga o t-SNE:** zniekształca odległości globalne i zależy od perplexity oraz ziarna, więc wnioskujemy o skupiskach lokalnych, nie o odległościach między skupiskami.

# Część B. Text Embeddings (2 pkt)

## B1. Osadzenia zdań, co najmniej 2 modele [0.5 pkt]

Trzy modele osadzeń zdań dla naszej anotowanej próbki:

- **FastText (korpus):** średnia wektorów wszystkich tokenów zdania. FastText daje wektor także dla słów spoza słownika (z n-gramów), więc używamy wszystkich tokenów.
- **Word2Vec (korpus):** średnia wektorów tokenów obecnych w słowniku (Word2Vec nie ma wektorów dla słów spoza słownika).
- **Universal Sentence Encoder:** model uczony na zdaniach. Dla polskiego potrzebna jest wersja wielojęzyczna (`universal-sentence-encoder-multilingual`), wymaga `tensorflow`, `tensorflow_hub` i `tensorflow_text` oraz dostępu do sieci. Jeśli nie da się go załadować, komórka go pomija, a pozostałe dwa modele spełniają wymóg "co najmniej 2 modele".

In [12]:
def sentence_vectors(kv, token_lists):
    """Średnia wektorów słów. Zwraca macierz i indeksy zdań, dla których udało się policzyć wektor."""
    use_oov = isinstance(kv, FastTextKeyedVectors)   # FastText: wektory z n-gramów także dla OOV
    vecs, keep = [], []
    for i, toks in enumerate(token_lists):
        v = [kv[w] for w in toks if use_oov or w in kv.key_to_index]
        if v:
            vecs.append(np.mean(v, axis=0))
            keep.append(i)
    return np.vstack(vecs), keep


token_lists = ann["tokens"].tolist()
SENT_EMB = {}    # nazwa modelu -> (macierz, indeksy zdań w ann)

for name, kv in ALL_MODELS.items():
    SENT_EMB[f"{name}, średnia"] = sentence_vectors(kv, token_lists)


# Universal Sentence Encoder (opcjonalnie)
# Teksty korpusu są stokenizowane ("Była m", "dobry ."). USE dostaje tekst
# odtokenizowany, żeby nie pogarszać wektorów.
def detokenize(t):
    t = re.sub(r"\s+([,.;:!?%)\]])", r"\1", t)
    t = re.sub(r"([(\[])\s+", r"\1", t)
    # sklejamy końcówki osobowe tylko po formach czasu przeszłego (-ł, -ła, -ło, -ły, -li),
    # np. "Była m" -> "Byłam", "mieli śmy" -> "mieliśmy"; inne słowa zostają bez zmian
    t = re.sub(r"\b([^\W\d_]*(?:ł|ła|ło|ły|li)) (m|ś|śmy|ście)\b", r"\1\2", t)
    return t


# tfhub.dev bywa już nieosiągalny; próbujemy kolejnych adresów (sprawdź aktualny
# adres modelu na Kaggle Models, jeśli żaden nie zadziała)
USE_URLS = [
    "https://tfhub.dev/google/universal-sentence-encoder-multilingual/3",
    "https://www.kaggle.com/models/google/universal-sentence-encoder/tensorFlow2/multilingual/2",
]
try:
    import tensorflow_hub as hub
    import tensorflow_text  # noqa: F401
    use_model, last_err = None, None
    for url in USE_URLS:
        try:
            use_model = hub.load(url)
            print(f"załadowano USE z {url}")
            break
        except Exception as e:
            last_err = e
            print(f"nie udało się: {url} ({type(e).__name__})")
    if use_model is None:
        raise last_err
    texts_use = ann["text"].apply(detokenize).tolist()
    E = np.vstack([use_model(texts_use[i:i + 256]).numpy() for i in range(0, len(texts_use), 256)])
    SENT_EMB["Universal Sentence Encoder (multilingual)"] = (E, list(range(len(ann))))
except Exception as e:
    print(f"pomijam USE ({type(e).__name__}: {str(e)[:150]})")

pd.DataFrame({n: {"zdań z wektorem": len(k), "wymiar": X.shape[1]} for n, (X, k) in SENT_EMB.items()}).T

pomijam USE (ModuleNotFoundError: No module named 'tensorflow_hub')


,zdań z wektorem,wymiar
"W2V (korpus), średnia",550,100
"FastText (korpus), średnia",550,100


* **Pełne pokrycie zdań:** Obie reprezentacje oparte na uśrednianiu wektorów słów (**Word2Vec** oraz **FastText**) pomyślnie wygenerowały osadzenia dla wszystkich **550 zdań** z zaanotowanej próbki.


* **Wymiarowość:** Wygenerowane macierze wektorów zdań dla obu modeli mają spójny wymiar równy **100**.


* **Spełnienie wymagań:** Obecność dwóch kompletnych reprezentacji uśrednionych spełnia formalny wymóg użycia co najmniej dwóch modeli osadzeń zdań w Części B.

## B2. t-SNE zdań dla każdego modelu, wykresy interaktywne [0.5 pkt]

Dla każdego modelu osobny wykres. Kolor to **nasza** etykieta, a w podpowiedzi widać tekst zdania, etykietę PolEmo (`gold`) i liczbę słów, więc łatwo znaleźć zdania, w których anotacja i korpus się różnią. Wykresy zapisują się do `anotacja/tsne_zdania_*.html`.

In [13]:
def plot_sentences(name, X, keep):
    sub = ann.iloc[keep].copy()
    XY = run_tsne(normalize(X))
    sub["x"], sub["y"] = XY[:, 0], XY[:, 1]
    sub["zdanie"] = sub["text"].str.slice(0, 140)
    sub["etykieta"] = sub["label"]
    sub["PolEmo"] = sub["gold"] if "gold" in sub else ""
    sub["liczba słów"] = sub["tokens"].apply(len)
    fig = px.scatter(
        sub, x="x", y="y", color="etykieta", color_discrete_map=COLORS,
        category_orders={"etykieta": SENT_LABELS},
        hover_name="zdanie", hover_data={"x": False, "y": False, "PolEmo": True, "liczba słów": True},
        title=f"t-SNE zdań: {name}", width=950, height=700,
    )
    fig.update_traces(marker=dict(size=8, opacity=0.8))
    fig.write_html(OUT / f"tsne_zdania_{safe_name(name)}.html")
    fig.show()


for name, (X, keep) in SENT_EMB.items():
    plot_sentences(name, X, keep)

* **Brak separacji polaryzacji (`POZ` vs `NEG`):** W obu modelach zdania pozytywne (zielone) i negatywne (czerwone) są ze sobą silnie przemieszane. Uśrednianie wektorów słów nie tworzy osobnych klastrów dla przeciwnego sentymentu.


* **Wyodrębnienie zdań neutralnych (`NEU`):** Zarówno dla Word2Vec (górna część wykresu, $y \in [10, 17]$), jak i FastText (prawa strona, $x \in [7, 18]$) zdania neutralne (szare) tworzą wyraźnie odseparowany podklaster. Wynika to z odmiennego leksykonu — zdania neutralne zawierają głównie opisy faktograficzne i pozbawione są leksyki wartościującej.


* **Rozproszenie zdań mieszanych (`AMB`):** Punkty pomarańczowe (`AMB`) są równomiernie rozproszone wewnątrz głównego skupiska, ponieważ uśrednienie wektorów słów o pozytywnej i negatywnej polaryzacji plasuje je w centrum przestrzeni emocjonalnej.


* **Główne przyczyny ograniczeń uśredniania:**
    * **Dominacja tematu nad emocją:** Wektor zdania jest zdominowany przez słowa rzeczownikowe i dziedzinowe (*pokój, lekarz, wizyta*), przez co zdania grupują się według tematyki, a nie ładunku emocjonalnego.
    * **Utrata struktury i negacji:** Uśrednianie gubi szyk zdań oraz wpływ modyfikatorów — zdanie *"naprawdę polecam"* i *"nie polecam"* po uśrednieniu wektorów lądują blisko siebie z uwagi na wspólne słowo kluczowe *polecam*.

## B3. Analiza osadzeń i dyskusja wg pytań pomocniczych [1 pkt]

Poniżej komórki, które dostarczają materiału do każdego z pięciu pytań z instrukcji. Dyskusję piszecie pod nimi.

### Wspólna baza: macierz podobieństw

In [14]:
# cosinus w każdym modelu liczony na wspólnym zbiorze zdań (te, które mają wektor we wszystkich modelach)
common = sorted(set.intersection(*[set(k) for _, k in SENT_EMB.values()]))
pos = {n: {idx: r for r, idx in enumerate(k)} for n, (_, k) in SENT_EMB.items()}
NORM = {n: normalize(X[[pos[n][i] for i in common]]) for n, (X, k) in SENT_EMB.items()}
SIM = {n: M @ M.T for n, M in NORM.items()}
sub_ann = ann.iloc[common].copy()
sub_ann["n_tok"] = sub_ann["tokens"].apply(len)
iu = np.triu_indices(len(common), k=1)
print(f"Wspólnych zdań: {len(common)}, par: {len(iu[0])}")

def show_pairs(name, idx_pairs, cols=("text", "label")):
    rows = []
    for i, j in idx_pairs:
        rows.append({"cos": round(float(SIM[name][i, j]), 3),
                     "zdanie 1": sub_ann.iloc[i]["text"][:110], "etykieta 1": sub_ann.iloc[i]["label"],
                     "zdanie 2": sub_ann.iloc[j]["text"][:110], "etykieta 2": sub_ann.iloc[j]["label"]})
    return pd.DataFrame(rows)

Wspólnych zdań: 550, par: 150975


**Pytanie 1: Czy są zdania, które model uznał za podobne, ale w rzeczywistości różnią się znaczeniem?**
Szukamy par o największym cosinusie, które mają **przeciwne** etykiety (POZ vs NEG) w naszej anotacji.

In [15]:
lab = sub_ann["label"].to_numpy()
for name in SIM:
    S = SIM[name][iu]
    opposite = ((lab[iu[0]] == "POZ") & (lab[iu[1]] == "NEG")) | ((lab[iu[0]] == "NEG") & (lab[iu[1]] == "POZ"))
    order = np.argsort(-np.where(opposite, S, -1))[:5]
    print(f"\n=== {name}: najbardziej podobne pary POZ vs NEG")
    display(show_pairs(name, [(iu[0][o], iu[1][o]) for o in order]))


=== W2V (korpus), średnia: najbardziej podobne pary POZ vs NEG


,cos,zdanie 1,etykieta 1,zdanie 2,etykieta 2
0,0.908,"skierował mnie na parking zewnętrzny 1 pomyłka , wybrał em się na obiadokolację a tu nie mam mnie na liście 2",NEG,"Przyjazd godz 22 Pan na recepcji już obrażony ze ktoś o tej porze zakłóca mu spokój , biorę klucz jadę na drug",POZ
1,0.902,"Babka potrafi nawet zapowiedziec z tygodnia na tydzien , ze bedzie pytac ta i ta grupe , wiec wylatywanie z za",NEG,mialem na 7.30 wiec nie uczestniczylem . bylem na ostatnich zajeciach i przyjalem 3 czyli cel zamierzony - hum,POZ
2,0.901,"Siatkowka Ocena : odradzam [ / b ] Jesli ktos chce miec do czynienia notorycznie z sytuacja "" maci e pilke",NEG,"urs : Analiza matematyczna , ćwiczenia Ocena : Nr 1 [ / b ] Jesli nie chcesz sie nameczyc , zaliczyc , to uder",POZ
3,0.896,Przed tym jednak zgłosiła m sie do dr Bocheńskiej ( jednej z najlepszych terapeutek jesli chodzi o problemy z,POZ,Już nie wspominając o 50 minutowym opóźnieniu i że lekarz spieszył się do domu a Pani z recepcji już z kimś ro,NEG
4,0.895,zamowilem pokoj a spalem w piwnicy cena byla kosmiczna za 3 dni ponad 600 zl nie polecam nikomu tego czegos os,NEG,mialem na 7.30 wiec nie uczestniczylem . bylem na ostatnich zajeciach i przyjalem 3 czyli cel zamierzony - hum,POZ



=== FastText (korpus), średnia: najbardziej podobne pary POZ vs NEG


,cos,zdanie 1,etykieta 1,zdanie 2,etykieta 2
0,0.929,"Siatkowka Ocena : odradzam [ / b ] Jesli ktos chce miec do czynienia notorycznie z sytuacja "" maci e pilke",NEG,"urs : Analiza matematyczna , ćwiczenia Ocena : Nr 1 [ / b ] Jesli nie chcesz sie nameczyc , zaliczyc , to uder",POZ
1,0.929,zamowilem pokoj a spalem w piwnicy cena byla kosmiczna za 3 dni ponad 600 zl nie polecam nikomu tego czegos os,NEG,mialem na 7.30 wiec nie uczestniczylem . bylem na ostatnich zajeciach i przyjalem 3 czyli cel zamierzony - hum,POZ
2,0.911,"Siatkowka Ocena : odradzam [ / b ] Jesli ktos chce miec do czynienia notorycznie z sytuacja "" maci e pilke",NEG,mialem na 7.30 wiec nie uczestniczylem . bylem na ostatnich zajeciach i przyjalem 3 czyli cel zamierzony - hum,POZ
3,0.910,"skierował mnie na parking zewnętrzny 1 pomyłka , wybrał em się na obiadokolację a tu nie mam mnie na liście 2",NEG,"Przyjazd godz 22 Pan na recepcji już obrażony ze ktoś o tej porze zakłóca mu spokój , biorę klucz jadę na drug",POZ
4,0.906,"Jestem zdecydowana na aparat i chciała m to zrobić sprawnie natomiast w tym gabinecie to niemożliwe , kiedy po",NEG,"Przyznam , że zawsze przed wizytami u poprzedniczki ( a uczęszczała m do jednego lekarza 8 lat ) była m W POZY",POZ


**Tak, obydwa modele przypisują bardzo wysokie podobieństwo cosinusowe ($\text{cos} \in [0{,}895; 0{,}929]$) zdaniom o skrajnie przeciwnym wydźwięku (`POZ` vs `NEG`)**.
**Przyczyny i wnioski**:

* **Dominacja słów ramowych i tematycznych:** Wektor średni jest zdominowany przez słowa opisujące dziedzinę (*lekarz, zajęcia, pokój*) oraz elementy szablonu recenzji (*ocena, jeśli, ktoś, na, w*). Pojedyncze słowa niosące sentyment (*odradzam*, *najlepszych*) giną w masie słów neutralnych.


* **Ograniczenie prostej średniej (Bag-of-Words):** Operacja uśredniania całkowicie ignoruje szyk zdań, strukturę składniową oraz obecność negacji.


* **Wyższe podobieństwo w FastText:** FastText osiąga jeszcze wyższe wartości cosinusa ($>0{,}92$) z powodu podsłów (n-gramów znakowych), które dodatkowo sztucznie zbliżają do siebie słowa o podobnych końcówkach gramatycznych i rdzeniach.

**Pytanie 2: Czy są zdania semantycznie podobne, ale model uznał je za odległe, bo użyto innych słów?**
Szukamy par o **tej samej** etykiecie, z bardzo małym wspólnym słownictwem (Jaccard tokenów < 0.1). Surowe cosinusy z różnych modeli nie są porównywalne (średnia wektorów ma cosinusy skupione wysoko, USE niżej), więc porównujemy **percentyle** cosinusa każdej pary w rozkładzie wszystkich par danego modelu. Pary, które USE uznaje za bliskie (wysoki percentyl), a średnia wektorów za dalekie, to dobre przykłady przewagi modelu zdaniowego. Ta sama etykieta i mało wspólnych słów to tylko kandydaci: przejrzyjcie pary ręcznie.

In [16]:
tok_sets = [set(t) for t in sub_ann["tokens"]]
def tok_jacc(i, j):
    a, b = tok_sets[i], tok_sets[j]
    return len(a & b) / len(a | b) if (a | b) else 0

rng = np.random.default_rng(SEED)
cand = rng.choice(len(iu[0]), size=min(200000, len(iu[0])), replace=False)
cand = [(iu[0][c], iu[1][c]) for c in cand
        if lab[iu[0][c]] == lab[iu[1][c]] and lab[iu[0][c]] in ("POZ", "NEG")
        and tok_jacc(iu[0][c], iu[1][c]) < 0.1]
print(f"Kandydatów (ta sama etykieta, mało wspólnych słów): {len(cand)}")

sorted_sims = {n: np.sort(SIM[n][iu]) for n in SIM}
def pct(name, p):
    return np.searchsorted(sorted_sims[name], SIM[name][p]) / len(sorted_sims[name])

use_name = next((n for n in SIM if n.startswith("Universal")), None)
other = [n for n in SIM if n != use_name]
if use_name and other and cand:
    ref = other[0]
    gap = sorted(cand, key=lambda p: -(pct(use_name, p) - pct(ref, p)))[:6]
    print(f"\n=== pary bliskie wg USE (wysoki percentyl), a dalekie wg '{ref}'")
    d = show_pairs(use_name, gap).rename(columns={"cos": "cos USE"})
    d["percentyl USE"] = [round(pct(use_name, p), 3) for p in gap]
    d[f"cos {ref}"] = [round(float(SIM[ref][p]), 3) for p in gap]
    d[f"percentyl {ref}"] = [round(pct(ref, p), 3) for p in gap]
    display(d)
else:
    for name in SIM:
        low = sorted(cand, key=lambda p: SIM[name][p])[:6]
        print(f"\n=== {name}: pary z tą samą etykietą i najniższym cosinusem")
        display(show_pairs(name, low))
print("UWAGA: ta sama etykieta + mało wspólnych słów to tylko kandydaci. "
      "Przejrzyj pary ręcznie i opisz w dyskusji te, które są naprawdę semantycznie bliskie.")

Kandydatów (ta sama etykieta, mało wspólnych słów): 37069

=== W2V (korpus), średnia: pary z tą samą etykietą i najniższym cosinusem


,cos,zdanie 1,etykieta 1,zdanie 2,etykieta 2
0,0.093,"English - just for You my Abi - best wishes , thank You for nice conversations and good time Hasim .",POZ,Każdego traktuje indywidualnie .,POZ
1,0.130,Stawia niesłuszne diagnozy .,NEG,Stęchlizna okropna odzież po powrocie musowo prać .,NEG
2,0.146,"Pokoje funkcjonalne , czyste .",POZ,Uwierzyła m mu : ( Po półtora roku łykania non stop leków przeciwzapalnych znowu zaczęła m szukać pomocy .,POZ
3,0.146,Stawia niesłuszne diagnozy .,NEG,"W pokoju zabaw dla dzieci brudno , wykładzina zapuszczona .",NEG
4,0.155,Stawia niesłuszne diagnozy .,NEG,"Tylko frytki , frytki itp .",NEG
5,0.159,Ceny na stronie internetowej nie przystają do kwot żądanych przez obsługę .,NEG,"( "" boli "" ) .",NEG



=== FastText (korpus), średnia: pary z tą samą etykietą i najniższym cosinusem


,cos,zdanie 1,etykieta 1,zdanie 2,etykieta 2
0,0.152,SZCZERZE NIE POLECAM ! ! !,NEG,"( "" boli "" ) .",NEG
1,0.200,"( "" boli "" ) .",NEG,Standard hotelu odbiega od luksusu .,NEG
2,0.202,Podsumowując - NIE polecam !,NEG,"( "" boli "" ) .",NEG
3,0.222,Ceny na stronie internetowej nie przystają do kwot żądanych przez obsługę .,NEG,"( "" boli "" ) .",NEG
4,0.238,Stawia niesłuszne diagnozy .,NEG,"Tylko frytki , frytki itp .",NEG
5,0.238,"( "" boli "" ) .",NEG,Za tę cenę spodziewał em się czegoś lepszego .,NEG


UWAGA: ta sama etykieta + mało wspólnych słów to tylko kandydaci. Przejrzyj pary ręcznie i opisz w dyskusji te, które są naprawdę semantycznie bliskie.


**Tak**, w przedstawionych wynikach występują zdania, które mimo zbieżnego sensu intencjonalnego lub tej samej polaryzacji emocjonalnej uzyskują bardzo niski współczynnik podobieństwa cosinusowego ($\text{cos} \in [0{,}093; 0{,}238]$) z powodu braku wspólnych słów (Jaccard < 0.1).

* **Brak wspólnego słownictwa (FastText, $\text{cos} = 0{,}238$):** Zdania *„SZCZERZE NIE POLECAM !”* i *„Za tę cenę spodziewał em się czegoś lepszego .”* pełnią tę samą funkcję (NEG), ale z braku wspólnych tokenów model uznaje je za słabo powiązane.
* **Różnice domenowe (Word2Vec, $\text{cos} = 0{,}146$):** Wypowiedzi z różnych branż o tej samej polaryzacji (np. medyczne *„Stawia niesłuszne diagnozy .”* vs hotelowe *„W pokoju... brudno .”*) lądują daleko od siebie z braku punktów wspólnych w słowniku.

* **Główna przyczyna (Ograniczenie prostej średniej):**
Uśrednianie wektorów słów (W2V / FastText) jest zdominowane przez leksykę tematyczną. Bez modeli uczonych bezpośrednio na całych zdaniach (jak Universal Sentence Encoder czy Transformer HerBERT) reprezentacja nie potrafi wychwycić abstrakcyjnej synonimii funkcjonalnej ani wspólnego wydźwięku, gdy słowa w zdaniach w ogóle się nie pokrywają.

**Pytanie 3: Czy zdania o różnej długości, ale podobnym sensie, są traktowane przez model podobnie?**
Porównujemy średni cosinus dla par tej samej etykiety w zależności od różnicy długości (liczby tokenów), oraz korelację cosinusa z różnicą długości.

**Zastrzeżenie:** "ta sama etykieta" jest tylko słabym przybliżeniem "podobnego sensu". Dwa zdania POZ mogą mówić o zupełnie różnych rzeczach (jedzenie vs obsługa), więc tabela pokazuje ogólny wpływ długości na cosinus wśród zdań o tej samej ocenie, a nie dowodzi, że model rozpoznaje parafrazy. Kilka par o dużej różnicy długości i dużym cosinusie warto więc przejrzeć ręcznie i opisać w dyskusji.

In [17]:
len_ = sub_ann["n_tok"].to_numpy()
dl = np.abs(len_[iu[0]] - len_[iu[1]])
same = lab[iu[0]] == lab[iu[1]]
bins = pd.cut(dl, [-1, 2, 5, 10, 20, 1000], labels=["0-2", "3-5", "6-10", "11-20", ">20"])
cols = {}
for name in SIM:
    S = SIM[name][iu]
    df_ = pd.DataFrame({"cos": S, "bin": bins, "same": same})
    s = df_[df_["same"]].groupby("bin", observed=True)["cos"].mean().round(3)
    s.index = s.index.astype(str)
    s["korelacja cos~|Δdł| (ta sama etykieta)"] = round(float(np.corrcoef(S[same], dl[same])[0, 1]), 3)
    cols[name] = s
print("Średni cosinus par o tej samej etykiecie wg różnicy długości (liczba tokenów):")
display(pd.DataFrame(cols))

Średni cosinus par o tej samej etykiecie wg różnicy długości (liczba tokenów):


,"W2V (korpus), średnia","FastText (korpus), średnia"
bin,,
0-2,0.585,0.629
3-5,0.591,0.636
6-10,0.602,0.646
11-20,0.617,0.661
>20,0.653,0.690
korelacja cos~|Δdł| (ta sama etykieta),0.202,0.206


* **Wzrost podobieństwa ze wzrostem różnicy długości:** Średni cosinus rośnie wraz ze wzrostem różnicy długości zdań — dla Word2Vec od 0,585 (różnica 0–2 tokenów) do 0,653 (różnica >20 tokenów), a dla FastText od 0,629 do 0,690.


* **Dodatnia korelacja:** Współczynnik korelacji między różnicą długości a podobieństwem cosinusowym jest dodatni i wynosi około 0,20 (0,202 dla W2V oraz 0,206 dla FastText).


* **Artefakt uśredniania wektorów:** Wynik ten pokazuje systemowe zniekształcenie metody uśredniania. Długie zdania zawierają więcej słów uniwersalnych i funkcyjnych, przez co ich wektory wypadkowe są ściągane w stronę środka przestrzeni wektorowej. W konsekwencji modele sztucznie zawyżają podobieństwo cosinusowe dla par o dużej różnicy długości, zamiast odzwierciedlać rzeczywistą relację semantyczną.

**Pytanie 4: Czy zauważasz różnicę w wartościach podobieństwa między krótkimi a długimi zdaniami? Czy długość wpływa na "wyrazistość" wektora?**
Sprawdzamy (a) średni cosinus do pozostałych zdań dla zdań krótkich i długich (kwartyle długości, w kolejności Q1..Q4), (b) normę wektora (przed normalizacją) w funkcji długości. Dla średniej wektorów można oczekiwać, że długie zdania są bliższe "środkowi" przestrzeni, czyli mniej wyraziste, ale trzeba to sprawdzić w tabeli.

In [18]:
q = pd.qcut(len_, 4, duplicates="drop")
q_names = np.array([f"Q{c + 1} {q.categories[c]}" for c in q.codes])
rows = {}
for name in SIM:
    S = SIM[name].copy()
    np.fill_diagonal(S, np.nan)
    mean_cos = np.nanmean(S, axis=1)
    X_raw = SENT_EMB[name][0][[pos[name][i] for i in common]]
    norms = np.linalg.norm(X_raw, axis=1)
    d_ = pd.DataFrame({"kwartyl długości": q_names, "średni cos do reszty": mean_cos, "norma wektora": norms})
    rows[name] = d_.groupby("kwartyl długości").mean().round(3)
display(pd.concat(rows, axis=1))
print("Uwaga: norma wektora USE jest zwykle bliska stałej (~1), więc dla USE ta kolumna raczej nie niesie "
      "informacji. Dla średniej wektorów porównajcie normę w Q1 i Q4: uśrednianie wielu słów może ją zmniejszać. "
      "Wnioski opierajcie na własnej tabeli.")

W2V (korpus), średnia                \
                  średni cos do reszty norma wektora   
kwartyl długości                                       
Q1 (0.999, 8.0]                  0.537         2.446   
Q2 (8.0, 12.0]                   0.587         2.179   
Q3 (12.0, 19.0]                  0.620         2.038   
Q4 (19.0, 85.0]                  0.661         1.904   

                 FastText (korpus), średnia                
                       średni cos do reszty norma wektora  
kwartyl długości                                           
Q1 (0.999, 8.0]                       0.583         2.733  
Q2 (8.0, 12.0]                        0.634         2.412  
Q3 (12.0, 19.0]                       0.666         2.273  
Q4 (19.0, 85.0]                       0.701         2.128

Uwaga: norma wektora USE jest zwykle bliska stałej (~1), więc dla USE ta kolumna raczej nie niesie informacji. Dla średniej wektorów porównajcie normę w Q1 i Q4: uśrednianie wielu słów może ją zmniejszać. Wnioski opierajcie na własnej tabeli.


* **Wzrost podobieństwa ze wzrostem różnicy długości:** Średni cosinus rośnie wraz ze wzrostem różnicy długości zdań — dla Word2Vec od 0,585 (różnica 0–2 tokenów) do 0,653 (różnica >20 tokenów), a dla FastText od 0,629 do 0,690.


* **Dodatnia korelacja:** Współczynnik korelacji między różnicą długości a podobieństwem cosinusowym jest dodatni i wynosi około 0,20 (0,202 dla W2V oraz 0,206 dla FastText).


* **Artefakt uśredniania wektorów:** Wynik ten pokazuje systemowe zniekształcenie metody uśredniania. Długie zdania zawierają więcej słów uniwersalnych i funkcyjnych, przez co ich wektory wypadkowe są ściągane w stronę środka przestrzeni wektorowej. W konsekwencji modele sztucznie zawyżają podobieństwo cosinusowe dla par o dużej różnicy długości, zamiast odzwierciedlać rzeczywistą relację semantyczną.

**Pytanie 5: Czy opracowane osadzenia zdań nadają się do wybranego zadania (klasyfikacja sentymentu, dla której prowadziliście anotację)?**
Ilościowo: regresja logistyczna na osadzeniach w walidacji krzyżowej (macro-F1, accuracy, F1 dla każdej klasy), powtórzonej dla 3 ziaren (średnia i odchylenie), z TF-IDF jako punktem odniesienia. Dla osadzeń cechy są standaryzowane (`StandardScaler` w pipeline, dopasowany tylko na foldzie treningowym), a regularyzacja jest silniejsza (`C=0.1`), bo wektory ze 100-512 wymiarami przy kilkuset zdaniach łatwo przeuczyć. TF-IDF jest dopasowywany wewnątrz foldów (pipeline), więc nie ma wycieku idf. Dodatkowo grupowanie KMeans (ARI/NMI) pokazuje, na ile struktura przestrzeni sama odtwarza etykiety bez nadzoru, a macierz pomyłek najlepszego modelu pokazuje, które klasy są mylone.

In [19]:
y_all = ann["label"].to_numpy()
SEEDS = [SEED, SEED + 1, SEED + 2]
print("Rozkład klas:", dict(pd.Series(y_all).value_counts()))


def make_skf(y, seed):
    min_class = pd.Series(y).value_counts().min()
    if min_class < 5:
        print(f"UWAGA: najmniejsza klasa ma {min_class} zdań (<5), CV niestabilne.")
    return StratifiedKFold(n_splits=max(2, min(5, int(min_class))), shuffle=True, random_state=seed)


def cv_eval(model, X, y):
    present = [l for l in SENT_LABELS if l in set(y)]
    f1s, accs, per = [], [], []
    for s in SEEDS:
        pred = cross_val_predict(model, X, y, cv=make_skf(y, s))
        f1s.append(f1_score(y, pred, average="macro"))
        accs.append((pred == y).mean())
        per.append(pd.Series(f1_score(y, pred, average=None, labels=present, zero_division=0), index=present))
    return (np.mean(f1s), np.std(f1s), np.mean(accs),
            pd.concat(per, axis=1).mean(axis=1).reindex(SENT_LABELS), pred)


def cluster_scores(X, y, k=4):
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(normalize(X))
    return adjusted_rand_score(y, km.labels_), normalized_mutual_info_score(y, km.labels_)


res, preds = {}, {}
lr_sparse = lambda: LogisticRegression(max_iter=2000, class_weight="balanced")
lr_dense = lambda: make_pipeline(StandardScaler(),
                                 LogisticRegression(max_iter=2000, class_weight="balanced", C=0.1))

# baseline TF-IDF (pipeline: vectorizer uczony tylko na foldzie treningowym)
tfidf_pipe = make_pipeline(
    TfidfVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None, min_df=2), lr_sparse())
f1m, f1s, acc, per, pred = cv_eval(tfidf_pipe, ann["text"].to_numpy(), y_all)
svd = TruncatedSVD(n_components=min(50, Xt.shape[1] - 1), random_state=SEED)
ari, nmi = cluster_scores(svd.fit_transform(Xt), y_all)
res["TF-IDF (baseline)"] = {"macro-F1": f1m, "± std": f1s, "accuracy": acc, "ARI (KMeans)": ari, "NMI (KMeans)": nmi,
                            **{f"F1 {l}": per[l] for l in SENT_LABELS}}
preds["TF-IDF (baseline)"] = (y_all, pred)

for name, (X, keep) in SENT_EMB.items():
    y_k = y_all[keep]
    f1m, f1s, acc, per, pred = cv_eval(lr_dense(), X, y_k)
    ari, nmi = cluster_scores(X, y_k)
    res[name] = {"macro-F1": f1m, "± std": f1s, "accuracy": acc, "ARI (KMeans)": ari, "NMI (KMeans)": nmi,
                 **{f"F1 {l}": per[l] for l in SENT_LABELS}}
    preds[name] = (y_k, pred)

res_df = pd.DataFrame(res).T
display(res_df.round(3))

best = res_df["macro-F1"].astype(float).idxmax()
print(f"\nMacierz pomyłek najlepszego modelu ({best}), ostatnie ziarno CV:")
yt, yp = preds[best]
display(pd.crosstab(pd.Series(yt, name="prawdziwa"), pd.Series(yp, name="przewidziana")
                    ).reindex(index=SENT_LABELS, columns=SENT_LABELS, fill_value=0))

Rozkład klas: {'NEG': np.int64(213), 'POZ': np.int64(177), 'NEU': np.int64(120), 'AMB': np.int64(40)}


C:\Users\pauko\anaconda3\envs\CONDA_ENVIRONMENT\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
C:\Users\pauko\anaconda3\envs\CONDA_ENVIRONMENT\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
C:\Users\pauko\anaconda3\envs\CONDA_ENVIRONMENT\Lib\site-packages\sklearn\cluster\_kmeans.py:1425: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(


,macro-F1,± std,accuracy,ARI (KMeans),NMI (KMeans),F1 POZ,F1 NEG,F1 NEU,F1 AMB
TF-IDF (baseline),0.391,0.018,0.435,0.014,0.010,0.478,0.496,0.343,0.247
"W2V (korpus), średnia",0.479,0.007,0.525,0.070,0.097,0.568,0.587,0.510,0.251
"FastText (korpus), średnia",0.458,0.002,0.519,0.062,0.082,0.565,0.583,0.523,0.159



Macierz pomyłek najlepszego modelu (W2V (korpus), średnia), ostatnie ziarno CV:


przewidziana,POZ,NEG,NEU,AMB
prawdziwa,,,,
POZ,97,28,27,25
NEG,28,116,38,31
NEU,16,30,64,10
AMB,12,8,4,16


Opracowane, uśrednione osadzenia zdań **tylko częściowo nadają się** do zadania klasyfikacji sentymentu: sprawdzają się jako zagęszczona reprezentacja w uczeniu z nadzorem (wyyraźnie przewyższają rzadki baseline TF-IDF), ale nie tworzą samodzielnych klastrów emocjonalnych w przestrzeni bez nadzoru.

* **Uczenie z nadzorem (Przewaga nad TF-IDF):**
* Najlepszy wynik w walidacji krzyżowej uzyskał model **Word2Vec (średnia)** z **macro-F1 = 0,479** oraz **accuracy = 0,525**, wyprzedzając **FastText (0,458)**.


* Oba osadzenia gęste znacząco poprawiły wynik względem punktu odniesienia **TF-IDF (macro-F1 = 0,391, accuracy = 0,435)**, co dowodzi, że wektory ciągłe lepiej uogólniają kontekst i relacje semantyczne w tekście.




* **Struktura bez nadzoru (KMeans):**
* Wskaźniki **ARI ($0{,}070$ dla W2V, $0{,}062$ dla FastText)** oraz **NMI ($0{,}097$ / $0{,}082$)** pozostają bliskie zera. Potwierdza to, że sama przestrzeń wektorowa nie grupuje zdań naturalnie pod kątem wydźwięku bez wsparcia klasyfikatora.




* **Analiza klas i macierz pomyłek (dla najlepszego modelu W2V):**
* **Klasy jednolite:** Model stosunkowo dobrze radzi sobie z wyrazistą polaryzacją: `NEG` (F1 = 0,587), `POZ` (F1 = 0,568) oraz z klasą `NEU` (F1 = 0,510).


* **Trudność z klasą `AMB`:** Najniższy wynik uzyskano dla wypowiedzi mieszanych `AMB` (F1 = 0,251 dla W2V i 0,159 dla FastText). Wynika to z małej liczności tej kategorii w próbie (40 zdań) oraz faktu, że uśrednienie cech dodatnich i ujemnych sprowadza wektor w obszar neutralny.


* **Główne pomyłki:** Macierz pomyłek wskazuje na częste błędy między przeciwnymi klasami `POZ` $\leftrightarrow$ `NEG` (po 28 błędnych zaklasyfikowań w obie strony) oraz `NEU` z `NEG` (30 zdań). Jest to bezpośrednia konsekwencja uśredniania wektorów słów, braku obsługi negacji oraz bliskości antonimów w modelu Skip-gram.

### Podsumowanie

* **Zdania podobne, ale różne znaczeniowo:** Uśrednianie promuje słowa tematyczne i szablony recenzji, ignorując ładunek emocjonalny i negację (np. FastText $\text{cos} = 0{,}929$ dla par `POZ` vs `NEG`).


* **Zdania podobne, ale odległe:** Brak wspólnych tokenów (Jaccard $<0{,}1$) drastycznie obniża podobieństwo wypowiedzi o tej samej intencji (np. FastText $\text{cos} = 0{,}238$ dla zdań krytycznych).
* **Wpływ długości na wektory:** Różnica długości zdań sztucznie zawyża cosinus (korelacja $r \approx 0{,}20$). Długie zdania (Q4) mają mniejszą normę wektora i są ściągane do środka przestrzeni, co rozmywa ich wyrazistość semantyczną.


* **Uczenie z nadzorem vs bez nadzoru:** Osadzenia gęste wyraźnie przewyższają baseline TF-IDF (macro-F1 = $0{,}391$). Najlepszy okazał się **Word2Vec (średnia)** z macro-F1 = $0{,}479$ i accuracy = $0{,}525$. Bez nadzoru (KMeans) przestrzeń nie grupuje zdań wg polaryzacji (ARI/NMI $\approx 0$).


* **Trudność klas i pomyłki:** Model poprawnie klasyfikuje `POZ`, `NEG` oraz `NEU` (F1 $> 0{,}50$), lecz całkowicie zawodzi na klasie mieszanej `AMB` (F1 = $0{,}251$) i często myli skrajne polaryzacje (`POZ` $\leftrightarrow$ `NEG`).


* **Rekomendacje:** Prosta średnia gubi kontekst, szyk zdań oraz wpływ negacji. W dalszych pracach należy wdrożyć lematyzację, retrofitting leksykonalny oraz przejść na głębokie osadzenia kontekstowe (np. HerBERT/PolBERT).